# Статистическая проверка результатов A/B-теста

## Цель анализа

В EDA видно, что test чаще приводит пользователя к контакту с продавцом. Но по сырым долям нельзя принимать продуктовые решения: разница может быть случайной.

Здесь проверяю основную метрику `has_contact` и несколько вторичных метрик. Данные уже прошли путь: генерация → SQLite → SQL-витрины → EDA → чистый user-level датасет.

## Загрузка данных

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.options.display.float_format = "{:.4f}".format

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from stat_tests import (
    bootstrap_mean_diff,
    format_money,
    format_p_value,
    format_percent,
    mann_whitney_test,
    proportion_z_test,
)

processed_dir = project_root / "data" / "processed"
presentation_dir = project_root / "presentation"

alpha = 0.05

data = pd.read_csv(processed_dir / "ab_test_ready_for_stats.csv")
print(f"Данные загружены: {data.shape[0]} строк, {data.shape[1]} столбцов")
display(data.head())


def format_pp(value):
    return f"{value * 100:.2f} п.п."


def display_metric_result(result):
    return pd.DataFrame([
        {
            "control": format_percent(result["control_rate"]),
            "test": format_percent(result["test_rate"]),
            "absolute_uplift": format_pp(result["absolute_difference"]),
            "relative_uplift": format_percent(result["relative_difference"]),
            "z_stat": f"{result['z_stat']:.4f}",
            "p_value": format_p_value(result["p_value"]),
        }
    ])

Данные загружены: 12000 строк, 12 столбцов


,user_id,experiment_group,has_contact,has_purchase,is_payer,revenue,views,contacts,purchases,region,device,user_segment
0,1,control,1,0,1,499.0000,5,1,0,Ural,android,regular
1,2,control,0,0,0,0.0000,8,0,0,Moscow,desktop,regular
2,3,control,1,0,0,0.0000,9,1,0,Siberia,mobile_web,bargain_hunter
3,4,control,0,0,0,0.0000,6,0,0,Moscow,android,new
4,5,control,0,0,0,0.0000,3,0,0,Moscow,desktop,new


## Валидация данных

Перед тестами проверяю, что датасет не сломан: нет пропусков, нет дублей пользователей, группы только `control` и `test`.

In [2]:
print("Размер датасета:", data.shape)
print("Количество пропусков:", int(data.isna().sum().sum()))
print("Дубли user_id:", int(data["user_id"].duplicated().sum()))
print("Группы:", sorted(data["experiment_group"].unique().tolist()))

group_sizes = data.groupby("experiment_group", as_index=False).agg(users_count=("user_id", "nunique"))
group_sizes["users_share"] = group_sizes["users_count"] / group_sizes["users_count"].sum()
group_sizes_display = group_sizes.copy()
group_sizes_display["users_share"] = group_sizes_display["users_share"].map(format_percent)
display(group_sizes_display)

binary_columns = ["has_contact", "has_purchase", "is_payer"]
for column in binary_columns:
    print(f"{column}: {sorted(data[column].unique().tolist())}")

assert data["user_id"].duplicated().sum() == 0, "В данных есть дубли пользователей"
assert data.isna().sum().sum() == 0, "В данных есть пропуски"
assert set(data["experiment_group"].unique()) == {"control", "test"}, "В данных есть неожиданные группы"
for column in binary_columns:
    assert set(data[column].unique()).issubset({0, 1}), f'В {column} есть значения кроме 0/1'


Размер датасета: (12000, 12)
Количество пропусков: 0
Дубли user_id: 0
Группы: ['control', 'test']


,experiment_group,users_count,users_share
0,control,6000,50.00%
1,test,6000,50.00%


has_contact: [0, 1]
has_purchase: [0, 1]
is_payer: [0, 1]


Группы по 6 000 пользователей, пропусков и дублей нет. Можно переходить к тестам.

## Гипотезы

Уровень значимости: `alpha = 0.05`.

**Основная метрика `has_contact`**

- H0: доля пользователей, связавшихся с продавцом, одинакова в control и test.
- H1: доля пользователей, связавшихся с продавцом, отличается между control и test.

**ARPU / revenue**

- H0: средняя выручка на пользователя одинакова в control и test.
- H1: средняя выручка на пользователя отличается между control и test.

In [3]:
control = data[data["experiment_group"] == "control"].copy()
test = data[data["experiment_group"] == "test"].copy()
control_total = len(control)
test_total = len(test)
print(f"control: {control_total} пользователей")
print(f"test: {test_total} пользователей")

control: 6000 пользователей
test: 6000 пользователей


## Основная метрика: has_contact

Для бинарной метрики использую двусторонний z-test для двух долей. Это прямой тест для вопроса: изменилась ли доля пользователей с контактом.

In [4]:
contact_result = proportion_z_test(
    control_successes=control["has_contact"].sum(),
    control_total=control_total,
    test_successes=test["has_contact"].sum(),
    test_total=test_total,
)

display(display_metric_result(contact_result))

if contact_result["p_value"] < alpha and contact_result["absolute_difference"] > 0:
    print("Вывод: test статистически значимо выше control по контактной конверсии.")
elif contact_result["p_value"] < alpha:
    print("Вывод: различие статистически значимо, но test не лучше control.")
else:
    print("Вывод: статистически значимого различия по has_contact не найдено.")

,control,test,absolute_uplift,relative_uplift,z_stat,p_value
0,28.60%,30.60%,2.00 п.п.,6.99%,2.3997,0.0164


Вывод: test статистически значимо выше control по контактной конверсии.


Контактная конверсия выросла с 28.60% до 30.60%. Абсолютный прирост — 2.00 п.п., p-value ниже 0.05. Для основной метрики это сильный сигнал в пользу test.

## Вторичные бинарные метрики

Проверяю `has_purchase` и `is_payer`. Эти метрики помогают увидеть, нет ли заметного побочного эффекта, но они не заменяют основную метрику.

In [5]:
binary_test_results = {}

for metric in ["has_purchase", "is_payer"]:
    result = proportion_z_test(
        control_successes=control[metric].sum(),
        control_total=control_total,
        test_successes=test[metric].sum(),
        test_total=test_total,
    )
    binary_test_results[metric] = result
    print(f"\nМетрика: {metric}")
    display(display_metric_result(result))
    if result["p_value"] < alpha:
        print("Различие статистически значимо.")
    else:
        print("Статистически значимого различия нет.")


Метрика: has_purchase


,control,test,absolute_uplift,relative_uplift,z_stat,p_value
0,1.95%,1.80%,-0.15 п.п.,-7.69%,-0.6057,0.5447


Статистически значимого различия нет.

Метрика: is_payer


,control,test,absolute_uplift,relative_uplift,z_stat,p_value
0,3.75%,3.63%,-0.12 п.п.,-3.11%,-0.3389,0.7347


Статистически значимого различия нет.


По покупкам и платежам значимого сдвига нет. Это нормально для вторичных и более редких событий: тест был нацелен прежде всего на контакт.

## Revenue / ARPU

Выручка разреженная: большинство пользователей ничего не платит. Из-за этого среднее может сильно шуметь, а t-test как единственная проверка был бы слабым выбором.

Использую bootstrap для разницы средних ARPU и дополнительно Mann-Whitney U test для revenue.

In [6]:
revenue_bootstrap = bootstrap_mean_diff(
    control_values=control["revenue"].values,
    test_values=test["revenue"].values,
    n_bootstrap=10000,
    random_state=42,
)
revenue_mann_whitney = mann_whitney_test(control["revenue"].values, test["revenue"].values)

revenue_display = pd.DataFrame([
    {
        "metric": "ARPU",
        "control": format_money(revenue_bootstrap["control_mean"]),
        "test": format_money(revenue_bootstrap["test_mean"]),
        "difference": format_money(revenue_bootstrap["observed_difference"]),
        "ci_95": f"[{format_money(revenue_bootstrap['ci_lower'])}; {format_money(revenue_bootstrap['ci_upper'])}]",
        "bootstrap_p_value": f"{revenue_bootstrap['bootstrap_p_value']:.6f}",
        "mann_whitney_p_value": f"{revenue_mann_whitney['p_value']:.6f}",
    }
])
display(revenue_display)

if revenue_bootstrap["bootstrap_p_value"] < alpha:
    print("По bootstrap ARPU статистически значимо отличается между группами.")
else:
    print("По bootstrap статистически значимого эффекта на ARPU нет.")

,metric,control,test,difference,ci_95,bootstrap_p_value,mann_whitney_p_value
0,ARPU,14.01,15.09,1.08,[-2.18; 4.40],0.527800,0.765537


По bootstrap статистически значимого эффекта на ARPU нет.


ARPU в test немного выше, но доверительный интервал включает 0, а p-value высокий. Денежный эффект не подтвержден статистически.

## Дополнительные количественные метрики: contacts и views

Проверяю среднее число контактов и просмотров на пользователя. Это помогает понять, растут ли контакты сами по себе или просто вместе с просмотрами.

In [7]:
continuous_results = {
    "revenue": {"bootstrap": revenue_bootstrap, "mann_whitney": revenue_mann_whitney}
}

for metric in ["contacts", "views"]:
    bootstrap_result = bootstrap_mean_diff(
        control_values=control[metric].values,
        test_values=test[metric].values,
        n_bootstrap=5000,
        random_state=42,
    )
    mw_result = mann_whitney_test(control[metric].values, test[metric].values)
    continuous_results[metric] = {"bootstrap": bootstrap_result, "mann_whitney": mw_result}

    result_display = pd.DataFrame([
        {
            "metric": metric,
            "control_mean": f"{bootstrap_result['control_mean']:.4f}",
            "test_mean": f"{bootstrap_result['test_mean']:.4f}",
            "difference": f"{bootstrap_result['observed_difference']:.4f}",
            "ci_95": f"[{bootstrap_result['ci_lower']:.4f}; {bootstrap_result['ci_upper']:.4f}]",
            "bootstrap_p_value": format_p_value(bootstrap_result["bootstrap_p_value"]),
            "mann_whitney_p_value": format_p_value(mw_result["p_value"]),
        }
    ])
    display(result_display)

,metric,control_mean,test_mean,difference,ci_95,bootstrap_p_value,mann_whitney_p_value
0,contacts,0.3432,0.3803,0.0372,[0.0157; 0.0595],0.001800,0.007614


,metric,control_mean,test_mean,difference,ci_95,bootstrap_p_value,mann_whitney_p_value
0,views,6.7698,6.7857,0.0158,[-0.1115; 0.1398],0.8138,0.7143


Среднее число контактов выше в test, а просмотры почти не отличаются. Это поддерживает продуктовую интерпретацию: изменение влияет именно на контакт, а не просто на общий объем активности.

## Статистическая и практическая значимость

Статистическая значимость говорит, что эффект трудно объяснить случайностью. Практическая значимость отвечает на другой вопрос: достаточно ли велик эффект для бизнеса.

In [8]:
primary_is_significant = contact_result["p_value"] < alpha
primary_is_positive = contact_result["absolute_difference"] > 0

if primary_is_significant and primary_is_positive:
    rollout_recommendation = "Можно рекомендовать rollout после проверки технических и бизнес-рисков."
elif primary_is_significant and not primary_is_positive:
    rollout_recommendation = "Rollout не рекомендуется: test статистически значимо хуже по основной метрике."
else:
    rollout_recommendation = "Rollout не стоит рекомендовать без дополнительной проверки."

print(f"Абсолютный uplift has_contact: {format_pp(contact_result['absolute_difference'])}")
print(f"Относительный uplift has_contact: {format_percent(contact_result['relative_difference'])}")
print(rollout_recommendation)

Абсолютный uplift has_contact: 2.00 п.п.
Относительный uplift has_contact: 6.99%
Можно рекомендовать rollout после проверки технических и бизнес-рисков.


Прирост 2.00 п.п. выглядит продуктово заметным для контактной метрики. Но перед rollout нужно проверить качество контактов: больше контактов не всегда означает больше хороших сделок.

## Summary table

In [9]:
def conclusion_text(p_value, absolute_difference):
    if p_value < alpha and absolute_difference > 0:
        return "статистически значимо выше в test"
    if p_value < alpha and absolute_difference < 0:
        return "статистически значимо ниже в test"
    return "статистически значимого различия не найдено"

summary_rows = []

summary_rows.append({
    "metric": "has_contact",
    "control_value": contact_result["control_rate"],
    "test_value": contact_result["test_rate"],
    "absolute_difference": contact_result["absolute_difference"],
    "relative_difference": contact_result["relative_difference"],
    "p_value": contact_result["p_value"],
    "is_significant": contact_result["p_value"] < alpha,
    "method": "two_proportion_z_test",
    "conclusion": conclusion_text(contact_result["p_value"], contact_result["absolute_difference"]),
})

for metric, result in binary_test_results.items():
    summary_rows.append({
        "metric": metric,
        "control_value": result["control_rate"],
        "test_value": result["test_rate"],
        "absolute_difference": result["absolute_difference"],
        "relative_difference": result["relative_difference"],
        "p_value": result["p_value"],
        "is_significant": result["p_value"] < alpha,
        "method": "two_proportion_z_test",
        "conclusion": conclusion_text(result["p_value"], result["absolute_difference"]),
    })

for metric in ["revenue", "contacts", "views"]:
    result = continuous_results[metric]["bootstrap"]
    control_value = result["control_mean"]
    test_value = result["test_mean"]
    absolute_difference = result["observed_difference"]
    relative_difference = absolute_difference / control_value if control_value != 0 else np.nan
    p_value = result["bootstrap_p_value"]

    summary_rows.append({
        "metric": metric,
        "control_value": control_value,
        "test_value": test_value,
        "absolute_difference": absolute_difference,
        "relative_difference": relative_difference,
        "p_value": p_value,
        "is_significant": p_value < alpha,
        "method": "bootstrap_mean_diff",
        "conclusion": conclusion_text(p_value, absolute_difference),
    })

summary = pd.DataFrame(summary_rows)
summary_path = processed_dir / "stat_test_results.csv"
summary.to_csv(summary_path, index=False)

summary_display = summary.copy()
for column in ["control_value", "test_value", "absolute_difference", "relative_difference"]:
    summary_display[column] = summary_display[column].map(lambda value: f"{value:.4f}")
summary_display["p_value"] = summary_display["p_value"].map(format_p_value)
display(summary_display)
print(f"Результаты сохранены: {summary_path.relative_to(project_root)}")

,metric,control_value,test_value,absolute_difference,relative_difference,p_value,is_significant,method,conclusion
0,has_contact,0.2860,0.3060,0.0200,0.0699,0.0164,True,two_proportion_z_test,статистически значимо выше в test
1,has_purchase,0.0195,0.0180,-0.0015,-0.0769,0.5447,False,two_proportion_z_test,статистически значимого различия не найдено
2,is_payer,0.0375,0.0363,-0.0012,-0.0311,0.7347,False,two_proportion_z_test,статистически значимого различия не найдено
3,revenue,14.0070,15.0883,1.0813,0.0772,0.5278,False,bootstrap_mean_diff,статистически значимого различия не найдено
4,contacts,0.3432,0.3803,0.0372,0.1083,0.001800,True,bootstrap_mean_diff,статистически значимо выше в test
5,views,6.7698,6.7857,0.0158,0.0023,0.8138,False,bootstrap_mean_diff,статистически значимого различия не найдено


Результаты сохранены: data\processed\stat_test_results.csv


## Финальный вывод

In [10]:
if primary_is_significant and primary_is_positive:
    final_contact_conclusion = "Тестовая версия статистически значимо улучшает контактную конверсию."
else:
    final_contact_conclusion = "По основной метрике нет достаточных оснований говорить об улучшении."

if revenue_bootstrap["bootstrap_p_value"] < alpha:
    final_revenue_conclusion = "По ARPU найдено статистически значимое различие, но его нужно отдельно проверить бизнес-контекстом."
else:
    final_revenue_conclusion = "По ARPU статистически значимого эффекта не найдено."

final_conclusion = f'''
{final_contact_conclusion}

{final_revenue_conclusion}

Рекомендация: {rollout_recommendation}

Ограничения: данные синтетические, эффект заложен в генератор, нет проверки качества контактов, жалоб, удержания и долгосрочной экономики. Перед реальным rollout нужно проверить корректность эксперимента, стабильность эффекта по сегментам и бизнес-ценность дополнительного контакта.
'''.strip()

print(final_conclusion)

Тестовая версия статистически значимо улучшает контактную конверсию.

По ARPU статистически значимого эффекта не найдено.

Рекомендация: Можно рекомендовать rollout после проверки технических и бизнес-рисков.

Ограничения: данные синтетические, эффект заложен в генератор, нет проверки качества контактов, жалоб, удержания и долгосрочной экономики. Перед реальным rollout нужно проверить корректность эксперимента, стабильность эффекта по сегментам и бизнес-ценность дополнительного контакта.


## Сохранение выводов

In [11]:
markdown_report = f'''# A/B-тест карточки объявления: SQL, EDA и статистическая проверка

## Контекст

Команда изменила карточку объявления и кнопку контакта с продавцом в учебном сервисе объявлений. Данные синтетические, проект образовательный. Главный вопрос: стала ли тестовая версия лучше доводить пользователя до контакта с продавцом.

## Гипотезы

Основная метрика — `has_contact`.

- H0: доля пользователей, связавшихся с продавцом, одинакова в control и test.
- H1: доля пользователей, связавшихся с продавцом, отличается между control и test.

Денежная метрика — `revenue` / ARPU.

- H0: средняя выручка на пользователя одинакова в control и test.
- H1: средняя выручка на пользователя отличается между control и test.

## Методы

- Для `has_contact`, `has_purchase` и `is_payer` использован двусторонний z-test для двух долей.
- Для ARPU использован bootstrap разницы средних с 95% доверительным интервалом.
- Для revenue дополнительно использован Mann-Whitney U test.
- Уровень значимости: {alpha}.

## Ключевые результаты

Основная метрика `has_contact`:

- control: {format_percent(contact_result['control_rate'])}
- test: {format_percent(contact_result['test_rate'])}
- абсолютный uplift: {format_pp(contact_result['absolute_difference'])}
- относительный uplift: {format_percent(contact_result['relative_difference'])}
- p-value: {contact_result['p_value']:.6f}

ARPU:

- control: {format_money(revenue_bootstrap['control_mean'])}
- test: {format_money(revenue_bootstrap['test_mean'])}
- разница test - control: {format_money(revenue_bootstrap['observed_difference'])}
- 95% bootstrap CI: [{format_money(revenue_bootstrap['ci_lower'])}; {format_money(revenue_bootstrap['ci_upper'])}]
- bootstrap p-value: {revenue_bootstrap['bootstrap_p_value']:.6f}
- Mann-Whitney p-value для revenue: {revenue_mann_whitney['p_value']:.6f}

## Рекомендация

Тестовая версия статистически значимо улучшает контактную конверсию. По ARPU статистически значимого эффекта не найдено: выручка разреженная, и среднее значение сильно шумит.

Rollout можно рекомендовать только после проверки технических и бизнес-рисков: корректности эксперимента, качества контактов, жалоб, стабильности эффекта по сегментам и стоимости внедрения.
'''

report_path = presentation_dir / "statistical_results.md"
report_path.write_text(markdown_report, encoding="utf-8")
print(f"Markdown-отчет сохранен: {report_path.relative_to(project_root)}")

Markdown-отчет сохранен: presentation\statistical_results.md
